# Model Building Phase - Logistic Regression
In this phase, we will combine Multivariate Analysis, feature engineering and Logistic Regression model to predict and infere the `Survived` from the independent variables. It will be an iterative process where we will make a Stable (By carefully meeting all assumptions) and Accurate (By doing intensive feature engineering and Multivariate Analysis) Logistic Regression Model.

## Creating the "Best dataset to Solve Age and Embarked missing value problem"
We need to handle Missing values in `Age` and `Embarked` columns. To do so, we will use Extrinsic approch. Create different models taking One-Constant-at-a-time and evaluate results on the basis of accuracy.

### Age column Imputation solution
We chose to take `Embarked` constant while finding the best `Age` imputation. Create seperate datasets for different Age imputations.

In [ ]:
import pandas as pd
import numpy as np
# ... other necessary imports

df = pd.read_csv('data/processed_titanic_dataset.csv')

#Creating a copy_df keeping One column for Embarked 
copy_df = df.copy()
copy_df = copy_df.drop(["Embarked_Missing_Imputing", "Embarked"], axis = 1)

In [ ]:
#One-Hot encoding and Standardization for Categorical and Numerical columns respectively
copy_df_encoded = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Total Family category", "Embarked_Mode_Imputing"], drop_first=True)

In [ ]:
#DataFrame with NaN values removed
copy_df_age_NaN_removing = copy_df_encoded.drop(["Age_Mean_Imputed", "Age_Median_Imputed"], axis = 1).dropna(subset = 'Age').reset_index(drop = True)

#Dataframe with mean value imputation
copy_df_age_Mean_imputation = copy_df_encoded.drop(["Age", "Age_Median_Imputed"], axis = 1)

#Dataframe with median value imputation
copy_df_age_Median_imputation = copy_df_encoded.drop(["Age_Mean_Imputed", "Age"], axis = 1)

#Dataframe with K-Imputer
from sklearn.impute import KNNImputer
copy_df_encoded_kNN_imputer = copy_df_encoded.drop(['Age_Mean_Imputed', 'Age_Median_Imputed'], axis = 1)
knn_imputer_object = KNNImputer(n_neighbors=5)
copy_df_encoded_kNN_imputer_imputed = knn_imputer_object.fit_transform(copy_df_encoded_kNN_imputer)
copy_df_encoded_kNN_imputer_imputed = pd.DataFrame(copy_df_encoded_kNN_imputer_imputed, columns=copy_df_encoded_kNN_imputer.columns)
copy_df_age_KNN_imputation = copy_df_encoded_kNN_imputer_imputed

In [ ]:
def model_building(df):
    #Training and testing split
    from sklearn.model_selection import train_test_split
    X = df.drop("Survived", axis=1)
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    #Standardizing data
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Model building
    from sklearn.linear_model import LogisticRegression
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)

    #Prediction
    y_pred = log_reg.predict(X_test_scaled)
    y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

    #Model evaluation
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score
    accuracy = accuracy_score(y_test, y_pred)
    confusion_matrix = confusion_matrix(y_test, y_pred)
    precision = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[1][0])
    recall = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[0][1])
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_prob)
    
    return accuracy, f1, auc_roc

#Checking the Accuray, AUC_ROC and F1 Score for all 3 datasets
print(model_building(copy_df_age_NaN_removing))
print(model_building(copy_df_age_Mean_imputation))
print(model_building(copy_df_age_Median_imputation))
print(model_building(copy_df_age_KNN_imputation))

Inference : Best Accuracy, AUC_ROC and F1 Score is attained by removing the NaN values from the dataset

#### Feature Engineering
Since, removing NaN values from the "Age" column provides highest accuracy, we will drop the 19% of the NaN columns from the dataset.

In [ ]:
df = df.drop(["Age_Mean_Imputed", "Age_Median_Imputed"], axis = 1)
df = df.dropna(subset = "Age").reset_index(drop = True)

### Embarked column Imputation solution
We took the featured engineered `Age` column and now we will find best imputation method for `Embarked`.

In [ ]:
#Creating a copy_df
copy_df = df.copy()

In [ ]:
#DataFrame with NaN values removed
copy_df_NaN_removing = copy_df.drop(["Embarked_Mode_Imputing", "Embarked_Missing_Imputing"], axis = 1).dropna(subset = "Embarked").reset_index(drop = True)
copy_df_NaN_removing = pd.get_dummies(copy_df_NaN_removing, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

#Dataframe with mean value imputation
copy_df_Mode_imputation = copy_df.drop(["Embarked", "Embarked_Missing_Imputing"], axis = 1)
copy_df_Mode_imputation = pd.get_dummies(copy_df_Mode_imputation, columns=["Pclass", "Sex", "Embarked_Mode_Imputing", "Total Family category"], drop_first=True)

#Dataframe with K-Imputer
from sklearn.impute import KNNImputer
copy_df_kNN_imputer = copy_df.drop(['Embarked_Mode_Imputing', 'Embarked_Missing_Imputing'], axis = 1)
copy_df_kNN_imputer = pd.get_dummies(copy_df_kNN_imputer, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)
knn_imputer_object = KNNImputer(n_neighbors=5)
copy_df_kNN_imputer_imputed = knn_imputer_object.fit_transform(copy_df_kNN_imputer)
copy_df_kNN_imputer_imputed = pd.DataFrame(copy_df_kNN_imputer_imputed, columns=copy_df_kNN_imputer.columns)
copy_df_KNN_imputation = copy_df_kNN_imputer_imputed

In [ ]:
#Checking the Accuray, AUC_ROC and F1 Score for all 3 datasets
print(model_building(copy_df_NaN_removing))
print(model_building(copy_df_Mode_imputation))
print(model_building(copy_df_KNN_imputation))

Inference : Best Accuracy, AUC_ROC and F1 Score is attained by removing the NaN values from the dataset.

#### Feature Engineering
Since, removing NaN values from the "Embarked" column provides highest accuracy, we will drop NaN rows from the dataset.

In [ ]:
df = df.drop(["Embarked_Mode_Imputing", "Embarked_Missing_Imputing"], axis = 1)
df = df.dropna(subset = "Embarked").reset_index(drop = True)

## Analysing the assumptions of Logistic Regression
Since now we have a cleaned dataset, we now need to check assumptions of Linear Regression on that dataset.

Assumptions of Logistic Regression Model are:
1. The dependent column (`Survived`) should be categorical with 2 categories.
2. There should be "No Multicollinarity" between the independent variables.
3. **There should be linear relationship between log odds and all independent variables. This assumption is specifically for Numerical Column `Age` and `Fare`. It is also applicable on Label Encoded independent variables ehich in our case out `Pclass`. In case of One-Hot encoded variables the assumption is always met.**
4. Independence of Errors.

The First 2 assumptions needs to be checked before model building phase and next 2 can be checked after model building phase.

### Assumption 1
Since the output column in `Survived` which is clearly categorical with 2 categories - First assumption is met for Logistic Regression.

### Assumption 2
According to the "No Multicollinearity" assumption, there should be No linear relationship between independent columns. To check the multicollinarity in the independent variables, we will perform 2 tests:-
1. Visual inspection using Pearson correlation and Heat Map. Peason tells the strength of linear relationship between the variables.
2. The confirmation will be taken on the basis of quantitative measurement using Variance Inflation Factor (VIF) which tells the strength of linear relationship of One independent variable with all others.

#### Visual Inspection

In [ ]:
copy_df = df.copy()
copy_df = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

import seaborn as sns 
import matplotlib.pyplot as plt 
corr = copy_df.corr() 
sns.heatmap(corr, annot=True, cmap="coolwarm") 
plt.show()

#### VIF method

In [ ]:
copy_df = df.copy()
copy_df = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
X = copy_df.drop(columns=["Survived"])
X_const = add_constant(X)
vif_data = pd.DataFrame()
vif_data["Feature"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) 
                   for i in range(X_const.shape[1])]

print(vif_data) #Since VIF values of all the columns are < 5, we can assume "No Multicollinarity"

Inference : 
- In visual Inspection all abs(values) are below  0.8, which confirms that "No Multicollinearity between independent variables".
- We confirmed by using VIF, which also checks for linear relationship between independent columns using regression analysis. Since all VIF's are less than 5, we confirm "No Multicollinearity between independent variables".

### Assumption 3 
To check this, we will first create a logistic regression model and then check the linearity of log odds with the numeric columns. We will do this using:
1. Visual Inspection.
2. Using Pearson correlation for log odds and independent numeric variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

copy_df = df.copy()
copy_df = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

def model_building(df):
    #Training and testing split
    from sklearn.model_selection import train_test_split
    X = df.drop("Survived", axis=1)
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    #Standardizing data
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Model building
    from sklearn.linear_model import LogisticRegression
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)

    #Prediction
    y_prob = log_reg.predict_proba(X_train_scaled)[:, 1]
    
    return y_prob, X_train

output_prob, X_train = model_building(copy_df)

In [ ]:
# To avoid division by zero or log(0), we can add a small epsilon
epsilon = 1e-10
log_odds_value = np.log((output_prob + epsilon) / (1 - (output_prob + epsilon)))

def plot_graph(x_column, logit_values, name_column):
    
    # Plotting the relationship
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=x_column, y=logit_values, alpha=0.5)
    sns.regplot(x=logit_values, y=logit_values, scatter=False, lowess=True, line_kws={'color': 'red'})

    plt.title(f"Linearity Check: {name_column} vs. Logit")
    plt.xlabel(name_column)
    plt.ylabel('Log-Odds (log(p/(1-p)))')
    plt.grid(True)
    plt.show()
    
plot_graph(X_train["Age"], log_odds_value, "Age")
plot_graph(X_train["Fare"], log_odds_value, "Fare")

In [ ]:
from scipy.stats import pearsonr
correlation_age, p_value_age = pearsonr(X_train["Age"], output_prob)
correlation_fare, p_value_fare = pearsonr(X_train['Fare'], output_prob)
print(p_value_age, p_value_fare)

Inference : 
With the help of visual inspection as well as pearson correlation, we are able to say the logit is significantly a linear relation of independent numeric columns.

## Creating the final model

In [ ]:
def model_building(df):
    #Training and testing split
    from sklearn.model_selection import train_test_split
    X = df.drop("Survived", axis=1)
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    #Standardizing data
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Model building
    from sklearn.linear_model import LogisticRegression
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)

    #Prediction
    y_pred = log_reg.predict(X_test_scaled)
    y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

    #Model evaluation
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score
    accuracy = accuracy_score(y_test, y_pred)
    confusion_matrix = confusion_matrix(y_test, y_pred)
    precision = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[1][0])
    recall = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[0][1])
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_prob)
    
    return accuracy, confusion_matrix, precision, recall, f1, auc_roc, log_reg, X.columns

df = pd.get_dummies(df, columns=["Pclass", "Sex", "Total Family category", "Embarked"], drop_first=True)
accuracy, confusion_matrix, precision, recall, f1, auc_roc, log_reg, input_features = model_building(df)

Inference:
- The model has fulfilled all the assumptions of the Logistic Regression model. 
- The model evaluation : 
    - Accuracy : 84.45%,
    -  confusion matrix : ([[73,  5],
      [16, 41]]),
    -  precision : 82.02%,
    -  recall : 93.58%,
    -  f1 score : 79.615,
    -  auc_roc score : 90.55%

# Model Inference - Logistic Regression

In [ ]:
# Create a DataFrame for interpretation
interpretation_df = pd.DataFrame({
    'Feature': input_features,
    'Coefficient': log_reg.coef_[0]
})

# Calculate Odds Ratio
interpretation_df['Odds Ratio'] = np.exp(interpretation_df['Coefficient'])

# Sort by Odds Ratio to see the most influential factors
interpretation_df = interpretation_df.sort_values(by='Odds Ratio', ascending=False)

## Final Model Interpretation: Key Factors for Survival on the Titanic : 
The logistic regression model has successfully identified several key factors that influenced a passenger's chances of survival. By converting the model's coefficients into Odds Ratios, we can clearly see the magnitude and direction of each factor's impact.

1. Sex (Male vs. Female):
- Odds Ratio for Sex_male: 0.318
- Inference: This is the most powerful predictor in the model. The odds of survival for a male passenger were only 0.318 times the odds for a female passenger, assuming all other factors (like class and age) were the same. To put it another way, a man's odds of surviving were about 68% lower than a woman's. This confirms the historical "women and children first" protocol.

2. Passenger Class (Pclass):
- Odds Ratio for Pclass_3: 0.359 (Compared to 1st Class)
- Odds Ratio for Pclass_2: 0.675 (Compared to 1st Class)
- Inference: Passenger class was the second most critical factor. The odds of survival for a passenger in 3rd class were about 64% lower than for a passenger in 1st class. The odds for a 2nd class passenger were 32.5% lower than for a 1st class passenger.
- Conclusion: There was a clear survival hierarchy based on wealth and status. First-class passengers had the highest chance of survival, followed by second, and then third class.

3. Age:
- Odds Ratio for Age: 0.535
- Inference: Age played a very significant role. For each additional year of a passenger's age, their odds of survival decreased by about 46.5% (1 - 0.535), holding all other factors constant. This strongly suggests that younger passengers had a much higher likelihood of surviving.

4. Family Size (Large vs. Alone):
- Odds Ratio for Total Family category_Large Family: 0.617
- Inference: Passengers traveling in a large family (more than 4 members) had 38% lower odds of survival compared to those traveling alone. This might be because it was more difficult for large families to stay together and evacuate quickly.

5. Fare:
- Odds Ratio for Fare: 1.103
- Inference: While less impactful than class, the fare paid still mattered. For every one-unit increase in fare (after standardization), a passenger's odds of survival increased by about 10%. This reinforces the idea that wealth was linked to survival, likely because higher fares corresponded to better cabin locations and access to lifeboats.

6. Point of Embarkation:
- Odds Ratio for Embarked_Q: 0.881 (Compared to Cherbourg)
- Odds Ratio for Embarked_S: 0.909 (Compared to Cherbourg)
- Inference: Passengers who boarded in Queenstown (Q) and Southampton (S) had slightly lower odds of survival (about 12% and 9% lower, respectively) compared to those who boarded in Cherbourg (C). This is a less direct factor and is likely correlated with other variables, such as the fact that a higher proportion of 1st class passengers boarded at Cherbourg.

7. Family Size (Small vs. Alone):
- Odds Ratio for Total Family category_Small Family: 0.995
- Inference: An odds ratio so close to 1 indicates that traveling in a small family (2-4 members) had almost no impact on survival odds compared to traveling alone.

### Summary of a Survivor's Profile
Based on the model, the passenger with the highest probability of surviving the Titanic disaster was a wealthy, young, female passenger from 1st class who boarded at Cherbourg and was not part of a large family. Conversely, the passenger with the lowest chance was an older, male passenger from 3rd class who was part of a large family.

In [ ]:
#Saving data for use of other models
df.to_csv('data/data_after_lr_model.csv', index=False)    